# 전처리 결과 확인

원본 CSV → 클리닝 → 이상치 처리 → die→unit 집계 결과를 확인하는 노트북

## 1. 환경 설정 및 데이터 로드

In [1]:
import os, sys

try:
    import google.colab
    if not os.path.exists("/content/project/setup.py"):
        os.system("pip install -q gdown")
        os.system("gdown --id 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip")
        os.system("unzip -qo /content/code.zip -d /content/project")
        os.makedirs("/content/project/0_data", exist_ok=True)
        os.system("gdown --id 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip")
        os.system("unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data")
        os.remove("/content/project/0_data/dataset.zip")
    if not os.path.exists("/content/project/2_preprocessing/cleaning.py"):
        os.system("gdown --id 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip")
        os.system("unzip -qo /content/preprocessing.zip -d /content/project")
    sys.path.insert(0, "/content/project")
    %run /content/project/setup.py
except ImportError:
    %run ../setup.py

# ─── 이 노트북에서 사용하는 import ───
from utils.config import PROJECT_ROOT
from utils.data import load_all, get_feat_cols, split_xs

# 전처리 모듈 import
sys.path.insert(0, os.path.join(PROJECT_ROOT, "2_preprocessing"))
from cleaning import run_cleaning
from outlier import run_outlier_treatment
from aggregation import run_aggregation

# 데이터 로드
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

setup 완료
Xs: (174980, 1091)  |  Ys: train=26,247, val=8,749, test=8,749


## 2. 클리닝 (상수/고결측/중복 제거 + 결측 imputation)

In [2]:
xs_train, xs_val, xs_test, clean_cols, clean_report = run_cleaning(
    xs, feat_cols, xs_dict,
    const_threshold=1e-6,      # std가 이 값 이하인 feature 제거
    missing_threshold=0.5,     # 결측률이 이 비율 이상인 feature 제거
    remove_duplicates=True,    # 값이 완전히 동일한 중복 컬럼 제거
)

클리닝 파이프라인 시작
원본 feature 수: 1087
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 982개
    컬럼: 1087 → 982 (105개 제거)
    DataFrame: (104988, 986)

[고결측 제거] threshold=50%
  제거: 5개, 잔여: 977개
    컬럼: 982 → 977 (5개 제거)
    DataFrame: (104988, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 26개, 잔여: 951개
    컬럼: 977 → 951 (26개 제거)
    DataFrame: (104988, 955)

[결측 imputation] train median 기준
  imputation 후 잔여 결측: 0

클리닝 완료: 1087 → 951 features (136개 제거)
  train: (104988, 955)
  val:   (34996, 955)
  test:  (34996, 955)


## 3. 이상치 처리 (Winsorization)

In [3]:
xs_train, xs_val, xs_test, outlier_report = run_outlier_treatment(
    xs_train, xs_val, xs_test, clean_cols,
    lower_pct=0.01,   # 하위 분위수 경계 (이 미만 값을 클리핑)
    upper_pct=0.99,   # 상위 분위수 경계 (이 초과 값을 클리핑)
)

이상치 처리 파이프라인 시작
[이상치 탐지] IQR × 1.5
  이상치 > 5%: 165개
  이상치 > 10%: 63개
[Winsorization] lower=1%, upper=99%
  적용 feature: 951개
[이상치 탐지] IQR × 1.5
  이상치 > 5%: 164개
  이상치 > 10%: 63개

이상치 처리 완료
  이상치 >5%  feature: 165 → 164 (1개 감소)
  이상치 >10% feature: 63 → 63 (0개 감소)
  train: (104988, 955)


## 4. Die → Unit 집계

In [4]:
unit_train, unit_val, unit_test, unit_feat_cols = run_aggregation(
    xs_train, xs_val, xs_test, clean_cols,
    agg_funcs=["mean", "std", "min", "max"],  # die 4개를 unit으로 집계할 통계량
    use_position_pivot=False,                   # True면 position별 피벗 feature도 추가
    save_csv=True,                              # 집계 결과를 CSV로 저장
)

Die → Unit 집계 시작
  agg_funcs: ['mean', 'std', 'min', 'max']
  position_pivot: False
집계 완료: 26,247 units × 3,804 features (agg: ['mean', 'std', 'min', 'max'])
집계 완료: 8,749 units × 3,804 features (agg: ['mean', 'std', 'min', 'max'])
집계 완료: 8,749 units × 3,804 features (agg: ['mean', 'std', 'min', 'max'])

집계 결과:
  train: (26247, 3804)
  val:   (8749, 3804)
  test:  (8749, 3804)
  feature 수: 3804

저장 완료 → c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output/unit_*.csv
